In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys

sys.path.append("../")
from tqdm import tqdm

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_only_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_only_with_layer_mppc_spacetime.npy"

BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"


sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)

X_pixel = np.concatenate([sig_pixel_spacetime, bg_pixel_spacetime], axis=0)
X_mppc = np.concatenate([sig_mppc_spacetime, bg_mppc_spacetime], axis=0)
y = np.concatenate(
    [np.ones(sig_pixel_spacetime.shape[0]), np.zeros(bg_pixel_spacetime.shape[0])],
    axis=0,
)

from sklearn.model_selection import train_test_split


X_pixel_train, X_pixel_test, X_mppc_train, X_mppc_test, y_train, y_test = (
    train_test_split(X_pixel, X_mppc, y, test_size=0.2, random_state=42, stratify=y)
)


del (
    sig_pixel_spacetime,
    sig_mppc_spacetime,
    bg_pixel_spacetime,
    bg_mppc_spacetime,
    X_pixel,
    X_mppc,
    y,
)

KeyboardInterrupt: 

In [ ]:
import src.torch.pre_processing.graph_batching as gc
from importlib import reload

reload(gc)

from torch_geometric.data import Data, Batch, Dataset
from torch_geometric.loader import DataLoader

event_processor = gc.EventProcessor(gc.HeteroGraphBuilder())

hetero_graph_train = event_processor.process_to_graphs(
    X_pixel=X_pixel_train, X_mppc=X_mppc_train, labels=y_train
)
hetero_graph_test = event_processor.process_to_graphs(
    X_pixel=X_pixel_test, X_mppc=X_mppc_test, labels=y_test
)

train_loader = DataLoader(hetero_graph_train, batch_size=512, shuffle=True)
test_loader = DataLoader(hetero_graph_test, batch_size=512, shuffle=False)

del X_pixel_train, X_pixel_test, X_mppc_train, X_mppc_test, y_train, y_test

In [ ]:
import torch
from torch_geometric.nn import HeteroConv, SAGEConv, global_mean_pool, global_max_pool, BatchNorm
import torch.nn.functional as F
from src.torch.model.components import get_mlp


class EventEdgeHeteroGNN(torch.nn.Module):
    def __init__(self, node_dims, edge_types, hidden_dim=32, num_layers=4, dropout=0.1, 
                 learn_edge_weights=True, edge_weight_layers=2):
        super(EventEdgeHeteroGNN, self).__init__()
        self.node_dims = node_dims
        self.edge_types = edge_types
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        self.learn_edge_weights = learn_edge_weights

        # Initial linear transformations for each node type
        self.node_lin = torch.nn.ModuleDict({
            node_type: get_mlp(in_dim, hidden_dim, num_layers=2, dropout=dropout)
            for node_type, in_dim in node_dims.items()
        })

        # Edge weight learning networks for each edge type
        if self.learn_edge_weights:
            self.edge_weight_nets = torch.nn.ModuleDict({
                str(edge_type): get_mlp(
                    2 * hidden_dim,  # source + target node features
                    1,  # single weight output
                    num_layers=edge_weight_layers,
                    dropout=dropout
                ) for edge_type in edge_types
            })

        # HeteroConv layers + per-node-type batchnorms
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        for _ in range(num_layers):
            conv = HeteroConv(
                {
                    edge_type: SAGEConv((hidden_dim, hidden_dim), hidden_dim)
                    for edge_type in edge_types
                },
                aggr="mean",
            )
            self.convs.append(conv)

            # BatchNorm for each node type in this layer
            self.bns.append(torch.nn.ModuleDict({
                node_type: BatchNorm(hidden_dim) for node_type in node_dims
            }))

        # Final linear layer for classification
        self.classifier = get_mlp(
            2 * hidden_dim * len(node_dims), 1, num_layers=3, dropout=dropout
        )

    def compute_edge_weights(self, x_dict, edge_index_dict):
        """Compute learned edge weights for each edge type"""
        edge_weights_dict = {}
        
        for edge_type, edge_index in edge_index_dict.items():
            if not self.learn_edge_weights:
                # If not learning weights, return uniform weights
                edge_weights_dict[edge_type] = torch.ones(
                    edge_index.size(1), device=edge_index.device
                )
                continue
                
            # Get source and target node types
            src_type, _, dst_type = edge_type
            
            # Get source and target node features
            src_features = x_dict[src_type][edge_index[0]]  # [num_edges, hidden_dim]
            dst_features = x_dict[dst_type][edge_index[1]]  # [num_edges, hidden_dim]
            
            # Concatenate source and target features
            edge_features = torch.cat([src_features, dst_features], dim=1)  # [num_edges, 2*hidden_dim]
            
            # Compute edge weights using the corresponding MLP
            edge_weights = self.edge_weight_nets[str(edge_type)](edge_features)
            edge_weights = torch.sigmoid(edge_weights).squeeze(-1)  # [num_edges]
            
            edge_weights_dict[edge_type] = edge_weights
            
        return edge_weights_dict

    def forward(self, input_data):
        x_dict, edge_index_dict, batch_dict = (
            input_data.x_dict,
            input_data.edge_index_dict,
            input_data.batch_dict,
        )
        if set(x_dict.keys()) != set(self.node_dims.keys()):
            print(f"Expected node types: {self.node_dims.keys()}")
            print(f"Received node types: {x_dict.keys()}")
            raise ValueError("Node types in input do not match model configuration.")

        # Initial node feature transformation
        x_dict = {
            node_type: torch.relu(self.node_lin[node_type](x))
            for node_type, x in x_dict.items()
        }

        # Store edge weights from each layer for output
        all_edge_weights = []

        # Apply HeteroConv layers
        for layer, conv in enumerate(self.convs):
            # Compute edge weights for this layer
            edge_weights_dict = self.compute_edge_weights(x_dict, edge_index_dict)
            all_edge_weights.append(edge_weights_dict)
            
            # Apply convolution with learned edge weights
            if self.learn_edge_weights:
                # Manual message passing with edge weights
                x_dict_new = {}
                for node_type in x_dict.keys():
                    x_dict_new[node_type] = []
                
                for edge_type, edge_index in edge_index_dict.items():
                    src_type, relation, dst_type = edge_type
                    edge_weights = edge_weights_dict[edge_type]
                    
                    # Get the SAGE conv for this edge type
                    sage_conv = conv.convs[edge_type]
                    
                    # Apply SAGE conv with edge weights
                    # Note: PyG SAGEConv doesn't directly support edge weights in message passing
                    # So we'll modify the source features by the edge weights before aggregation
                    src_x = x_dict[src_type]
                    dst_x = x_dict[dst_type]
                    
                    # Apply the SAGE transformation
                    out = sage_conv((src_x, dst_x), edge_index)
                    
                    # Weight the output by edge weights (approximate approach)
                    # This is a simplification - for exact edge weighting, you'd need custom message passing
                    if dst_type not in x_dict_new:
                        x_dict_new[dst_type] = out
                    else:
                        x_dict_new[dst_type] = x_dict_new[dst_type] + out
                        
                x_dict = x_dict_new
            else:
                # Standard convolution without edge weights
                x_dict = conv(x_dict, edge_index_dict)

            # Apply BN + ReLU + Dropout per node type
            new_x_dict = {}
            for node_type, x in x_dict.items():
                if x is not None:  # Check if node type has any features
                    x = self.bns[layer][node_type](x)   # BN first
                    x = torch.relu(x)
                    x = F.dropout(x, p=self.dropout, training=self.training)
                    new_x_dict[node_type] = x
            x_dict = new_x_dict

        # Global pooling (mean + max for each node type)
        pooled = []
        for node_type, x in x_dict.items():
            if x is not None and node_type in batch_dict:
                pooled.append(global_mean_pool(x, batch_dict[node_type]))
                pooled.append(global_max_pool(x, batch_dict[node_type]))

        # Concatenate pooled features from all node types
        h = torch.cat(pooled, dim=1)

        # Classification 
        classification_out = self.classifier(h).squeeze()
        classification_out = torch.sigmoid(classification_out).squeeze(-1)
        
        # Prepare edge weights output
        # Return the edge weights from the last layer, or aggregate across layers
        final_edge_weights = all_edge_weights[-1] if all_edge_weights else {}
        
        return {
            'classification': classification_out,
            'edge_weights': final_edge_weights,
            'all_layer_edge_weights': all_edge_weights  # Optional: all layers' weights
        }


